In [ ]:
import numpy as np

from scipy.integrate import solve_ivp
from scipy import sparse
from scipy.sparse.linalg import eigs

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error

import plotly.graph_objects as go

import optuna
import optuna.visualization as vis

In [ ]:
time = 100
dt = 0.005
steps = int(time / dt)

time = 100
dt = 0.005

steps = int(time / dt)
transient_steps_lorentz = int(steps * 0.1)
transient_steps_springs = int(steps * 0.1)
total_steps = steps + transient_steps_lorentz + transient_steps_springs

test_size = 0.2

t = np.linspace(0, total_steps * dt, total_steps)

# Init

In [3]:
sigma, rho, beta = 10, 28, 8.0 / 3.0

initial_state = [1.0, 1.0, 1.0]

def lorenz_system(t, state, sigma=sigma, rho=rho, beta=beta):
    x, y, z = state
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return [dx, dy, dz]

sol = solve_ivp(
    lorenz_system,
    (0, total_steps * dt),
    initial_state,
    t_eval=t,
)

lorentz_dataset = sol.y.T[transient_steps_lorentz:]

In [4]:
def lorentz_plot(data):
    fig = go.Figure(
        data=go.Scatter3d(
            x=data[:, 0],
            y=data[:, 1],
            z=data[:, 2],
            mode="lines",
            line=dict(color="blue", width=2),
        )
    )
    return fig

In [5]:
fig = lorentz_plot(lorentz_dataset)
fig.show()

In [6]:
def r_2_plot(actual, predicted):
    r_2 = r2_score(actual, predicted)
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=actual, y=predicted,
        mode='markers',
        name='Data Points',
        marker=dict(color='rgba(50, 50, 200, 0.5)', size=5)
    ))

    min_val = min(actual.min(), predicted.min())
    max_val = max(actual.max(), predicted.max())
    fig.add_trace(go.Scatter(
        x=[min_val, max_val], y=[min_val, max_val],
        mode='lines',
        name='Ideal Prediction',
        line=dict(color='firebrick', dash='dash')
    ))

    fig.update_layout(
        title=f"Actual vs. Predicted (R²: {r_2:.4f})",
        xaxis_title="Actual Values",
        yaxis_title="Predicted Values",
        showlegend=True
    )
    return fig

# Training

In [370]:
in_size = 3 # W_in
out_size = (
    3  # In and out 3 since we have 3D data and next position output is also a 3D pos
)
res_size = 500 # W_res
sparsity = 0.05
spec_rad = 1.2  # Need to test this out tbh
alpha = 0.3  # 70 percent previous and 30 percent current - Like adam momentum nn methods
tau_steps = 1

In [400]:
rng = np.random.default_rng(42)

bias = rng.uniform(-0.1, 0.1, res_size)

W_in = rng.uniform(
    -0.1, 0.1, (res_size, in_size)
)  # Splits the 3 inputs to a 100 dims

W_sparse = sparse.random(
    res_size,
    res_size,
    density=sparsity,
    format="csr",
    data_rvs=lambda n: rng.uniform(-1.0, 1.0, n),
    rng=rng,
)  # CSR is Compressed Sparse Row format also basically a Erdős–Rényi graph

eigenvalues, _ = eigs(W_sparse, k=1, which="LM")
largest_eigenvalue = np.abs(eigenvalues[0])

W_res = W_sparse * (spec_rad / largest_eigenvalue) if largest_eigenvalue > 0 else W_sparse

In [401]:
scaler = StandardScaler()
lorentz_scaled = scaler.fit_transform(lorentz_dataset)

In [402]:
X = np.zeros((steps + transient_steps_springs, res_size))
X[0] = np.tanh(0)

for i in range(1, steps + transient_steps_springs):
    u = lorentz_scaled[i - 1]  # Current input
    prev_state = X[i - 1]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X[i] = (1 - alpha) * prev_state + alpha * new_state

X = X[transient_steps_springs:]

In [ ]:
X_model_data = X[:-tau_steps]
Y_model_data = lorentz_scaled[transient_steps_springs + tau_steps:]

X_train, X_test = (
    X_model_data[:int(len(X_model_data) * (1 - test_size))],
    X_model_data[int(len(X_model_data) * (1 - test_size)):],
)
Y_train, Y_test_unscaled = (
    Y_model_data[:int(len(Y_model_data) * (1 - test_size))],
    Y_model_data[int(len(Y_model_data) * (1 - test_size)):],
)

model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)

Y_pred = scaler.inverse_transform(Y_pred_scaled)
Y_test = scaler.inverse_transform(Y_test_unscaled)

Optimal Alpha: 0.001


In [ ]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

1.0000 0.0079


## Plots

In [ ]:
fig = lorentz_plot(lorentz_dataset[-Y_pred.shape[0]:])
fig.show()
fig = lorentz_plot(Y_pred)
fig.show()

In [406]:
fig = r_2_plot(Y_test[:, 0], Y_pred[:, 0])
fig.show()
fig = r_2_plot(Y_test[:, 1], Y_pred[:, 1])
fig.show()
fig = r_2_plot(Y_test[:, 2], Y_pred[:, 2])
fig.show()

# Open Loop

In [407]:
steps_to_predict = int(steps * test_size)
current_step = steps - steps_to_predict - 1

current_X = X[current_step]

Y_pred_open = np.zeros((steps_to_predict, out_size))
Y_pred_open[0] = scaler.inverse_transform(model.predict(current_X.reshape(1, -1)))

for i in range(1, steps_to_predict):
    u_raw = Y_pred_open[i - 1]
    u_scaled = scaler.transform(u_raw.reshape(1, -1)).flatten()

    prev_state = current_X
    new_state = np.tanh(W_in @ u_scaled + W_res @ prev_state + bias)
    current_X = (1 - alpha) * prev_state + alpha * new_state

    Y_pred_open[i] = scaler.inverse_transform(model.predict(current_X.reshape(1, -1)))

In [408]:
r_2 = r2_score(Y_test, Y_pred_open)
mse = root_mean_squared_error(Y_test, Y_pred_open)

print(f"{r_2:.4f}", f"{mse:.4f}")

-2.1119 15.0108


## Plots

In [ ]:
fig = lorentz_plot(lorentz_dataset[-Y_pred.shape[0]:])
fig.show()
fig = lorentz_plot(Y_pred_open)
fig.show()

In [410]:
fig = r_2_plot(Y_test[:, 0], Y_pred_open[:, 0])
fig.show()
fig = r_2_plot(Y_test[:, 1], Y_pred_open[:, 1])
fig.show()
fig = r_2_plot(Y_test[:, 2], Y_pred_open[:, 2])
fig.show()

# Bayesian Optimization

## Closed

In [ ]:
scaler = StandardScaler()
lorentz_scaled = scaler.fit_transform(lorentz_dataset)

def objective(trial):
    res_size = trial.suggest_int("res_size", 50, 500)
    sparsity = trial.suggest_float("sparsity", 0.05, 0.5)
    spec_rad = trial.suggest_float("spec_rad", 0.5, 2.0)
    alpha = trial.suggest_float("alpha", 0.1, 0.8)

    in_size = 3
    out_size = 3
    tau_steps = 1

    rng = np.random.default_rng(42)
    bias = rng.uniform(-0.1, 0.1, res_size)
    W_in = rng.uniform(-0.1, 0.1, (res_size, in_size))
    W_sparse = sparse.random(
        res_size,
        res_size,
        density=sparsity,
        format="csr",
        data_rvs=lambda n: rng.uniform(-1.0, 1.0, n),
        rng=rng,
    )
    eigenvalues, _ = eigs(W_sparse, k=1, which="LM")
    largest_eigenvalue = np.abs(eigenvalues[0])
    W_res = (
        W_sparse * (spec_rad / largest_eigenvalue)
        if largest_eigenvalue > 0
        else W_sparse
    )

    X = np.zeros((steps + transient_steps_springs, res_size))
    X[0] = np.tanh(0)
    for i in range(1, steps + transient_steps_springs):
        u = lorentz_scaled[i - 1]
        prev_state = X[i - 1]
        new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
        X[i] = (1 - alpha) * prev_state + alpha * new_state
    X = X[transient_steps_springs:]

    X_model_data = X[:-tau_steps]
    Y_model_data = lorentz_scaled[transient_steps_springs + tau_steps:]
    X_train, X_test = (
        X_model_data[:int(len(X_model_data) * (1 - test_size))],
        X_model_data[int(len(X_model_data) * (1 - test_size)):],
    )
    Y_train, Y_test_unscaled = (
        Y_model_data[:int(len(Y_model_data) * (1 - test_size))],
        Y_model_data[int(len(Y_model_data) * (1 - test_size)):],
    )
    model = RidgeCV()
    model.fit(X_train, Y_train)
    Y_pred_scaled = model.predict(X_test)
    Y_pred = scaler.inverse_transform(Y_pred_scaled)
    Y_test = scaler.inverse_transform(Y_test_unscaled)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return r_2, mse

In [427]:
study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=50)

In [430]:
best_trials = study.best_trials

print("Pareto Front trials:")
for trial in best_trials:
    print(f"Trial {trial.number}: R2={trial.values[0]}, MSE={trial.values[1]}")
    print(f"Params: {trial.params}")

Pareto Front trials:
Trial 0: R2=0.9999981621744678, MSE=0.011126749913362583
Params: {'res_size': 467, 'sparsity': 0.36077537676760907, 'spec_rad': 1.0315171731208905, 'alpha': 0.7469881883161585}


In [433]:
best_params = study.best_trials[0].params

res_size = best_params["res_size"]
sparsity = best_params["sparsity"]
spec_rad = best_params["spec_rad"]
alpha = best_params["alpha"]

in_size = 3
out_size = 3
tau_steps = 1

rng = np.random.default_rng(42)
bias = rng.uniform(-0.1, 0.1, res_size)
W_in = rng.uniform(-0.1, 0.1, (res_size, in_size))
W_sparse = sparse.random(
    res_size,
    res_size,
    density=sparsity,
    format="csr",
    data_rvs=lambda n: rng.uniform(-1.0, 1.0, n),
    rng=rng,
)
eigenvalues, _ = eigs(W_sparse, k=1, which="LM")
largest_eigenvalue = np.abs(eigenvalues[0])
W_res = (
    W_sparse * (spec_rad / largest_eigenvalue)
    if largest_eigenvalue > 0
    else W_sparse
)

X = np.zeros((steps + transient_steps_springs, res_size))
X[0] = np.tanh(0)
for i in range(1, steps + transient_steps_springs):
    u = lorentz_scaled[i - 1]  # Current input
    prev_state = X[i - 1]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X[i] = (1 - alpha) * prev_state + alpha * new_state
X = X[transient_steps_springs:]

X_model_data = X[:-tau_steps]
Y_model_data = lorentz_scaled[transient_steps_springs + tau_steps:]
X_train, X_test = (
    X_model_data[:int(len(X_model_data) * (1 - test_size))],
    X_model_data[int(len(X_model_data) * (1 - test_size)):],
)
Y_train, Y_test_unscaled = (
    Y_model_data[:int(len(Y_model_data) * (1 - test_size))],
    Y_model_data[int(len(Y_model_data) * (1 - test_size)):],
)
model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)
Y_pred = scaler.inverse_transform(Y_pred_scaled)
Y_test = scaler.inverse_transform(Y_test_unscaled)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(r_2, mse)

0.9999981621744646 0.011126749924820918


### Plots

In [434]:
fig = lorentz_plot(lorentz_dataset[-Y_pred.shape[0]:])
fig.show()
fig = lorentz_plot(Y_pred)
fig.show()

In [435]:
fig = r_2_plot(Y_test[:, 0], Y_pred[:, 0])
fig.show()
fig = r_2_plot(Y_test[:, 1], Y_pred[:, 1])
fig.show()
fig = r_2_plot(Y_test[:, 2], Y_pred[:, 2])
fig.show()

In [437]:
vis.plot_pareto_front(study).show()

## Open

In [474]:
scaler = StandardScaler()
lorentz_scaled = scaler.fit_transform(lorentz_dataset)

def objective(trial):
    res_size = trial.suggest_int("res_size", 50, 500)
    sparsity = trial.suggest_float("sparsity", 0.05, 0.5)
    spec_rad = trial.suggest_float("spec_rad", 0.5, 2.0)
    alpha = trial.suggest_float("alpha", 0.1, 0.8)

    in_size = 3
    out_size = 3
    tau_steps = 1

    rng = np.random.default_rng(42)
    bias = rng.uniform(-0.1, 0.1, res_size)
    W_in = rng.uniform(-0.1, 0.1, (res_size, in_size))
    W_sparse = sparse.random(
        res_size,
        res_size,
        density=sparsity,
        format="csr",
        data_rvs=lambda n: rng.uniform(-1.0, 1.0, n),
        rng=rng,
    )
    try:
        eigenvalues, _ = eigs(W_sparse, k=1, which="LM")
    except Exception as e:
        raise optuna.exceptions.TrialPruned()
    largest_eigenvalue = np.abs(eigenvalues[0])
    W_res = (
        W_sparse * (spec_rad / largest_eigenvalue)
        if largest_eigenvalue > 0
        else W_sparse
    )

    X = np.zeros((steps + transient_steps_springs, res_size))
    X[0] = np.tanh(0)
    for i in range(1, steps + transient_steps_springs):
        u = lorentz_scaled[i - 1]
        prev_state = X[i - 1]
        new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
        X[i] = (1 - alpha) * prev_state + alpha * new_state
    X = X[transient_steps_springs:]

    X_model_data = X[:-tau_steps]
    Y_model_data = lorentz_scaled[transient_steps_springs + tau_steps :]
    X_train, X_test = (
        X_model_data[: int(len(X_model_data) * (1 - test_size))],
        X_model_data[int(len(X_model_data) * (1 - test_size)) :],
    )
    Y_train, Y_test_unscaled = (
        Y_model_data[: int(len(Y_model_data) * (1 - test_size))],
        Y_model_data[int(len(Y_model_data) * (1 - test_size)) :],
    )
    model = RidgeCV()
    model.fit(X_train, Y_train)
    Y_pred_scaled = model.predict(X_test)
    Y_pred = scaler.inverse_transform(Y_pred_scaled)
    Y_test = scaler.inverse_transform(Y_test_unscaled)

    steps_to_predict = int(steps * test_size)
    current_step = steps - steps_to_predict - 1
    current_X = X[current_step]
    Y_pred_open = np.zeros((steps_to_predict, out_size))
    Y_pred_open[0] = scaler.inverse_transform(model.predict(current_X.reshape(1, -1)))
    for i in range(1, steps_to_predict):
        u_raw = Y_pred_open[i - 1]
        u_scaled = scaler.transform(u_raw.reshape(1, -1)).flatten()
        prev_state = current_X
        new_state = np.tanh(W_in @ u_scaled + W_res @ prev_state + bias)
        current_X = (1 - alpha) * prev_state + alpha * new_state
        Y_pred_open[i] = scaler.inverse_transform(model.predict(current_X.reshape(1, -1)))

    r_2 = r2_score(Y_test, Y_pred_open)
    mse = root_mean_squared_error(Y_test, Y_pred_open)

    return mse

In [475]:
study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=50)

In [477]:
# best_trials = study.best_trials

# print("Pareto Front trials:")
# for trial in best_trials:
#     print(f"Trial {trial.number}: R2={trial.values[0]}, MSE={trial.values[1]}")
#     print(f"Params: {trial.params}")

In [480]:
study.best_params

{'res_size': 57,
 'sparsity': 0.14877861196431202,
 'spec_rad': 1.8410695149186684,
 'alpha': 0.6488495442516511}

In [493]:
best_params = study.best_params

res_size = best_params["res_size"]
sparsity = best_params["sparsity"]
spec_rad = best_params["spec_rad"]
alpha = best_params["alpha"]

in_size = 3
out_size = 3
tau_steps = 1

rng = np.random.default_rng(42)
bias = rng.uniform(-0.1, 0.1, res_size)
W_in = rng.uniform(-0.1, 0.1, (res_size, in_size))
W_sparse = sparse.random(
    res_size,
    res_size,
    density=sparsity,
    format="csr",
    data_rvs=lambda n: rng.uniform(-1.0, 1.0, n),
    rng=rng,
)
eigenvalues, _ = eigs(W_sparse, k=1, which="LM")
largest_eigenvalue = np.abs(eigenvalues[0])
W_res = (
    W_sparse * (spec_rad / largest_eigenvalue)
    if largest_eigenvalue > 0
    else W_sparse
)

X = np.zeros((steps + transient_steps_springs, res_size))
X[0] = np.tanh(0)
for i in range(1, steps + transient_steps_springs):
    u = lorentz_scaled[i - 1]
    prev_state = X[i - 1]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X[i] = (1 - alpha) * prev_state + alpha * new_state
X = X[transient_steps_springs:]

X_model_data = X[:-tau_steps]
Y_model_data = lorentz_scaled[transient_steps_springs + tau_steps :]
X_train, X_test = (
    X_model_data[: int(len(X_model_data) * (1 - test_size))],
    X_model_data[int(len(X_model_data) * (1 - test_size)) :],
)
Y_train, Y_test_unscaled = (
    Y_model_data[: int(len(Y_model_data) * (1 - test_size))],
    Y_model_data[int(len(Y_model_data) * (1 - test_size)) :],
)
model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)
Y_pred = scaler.inverse_transform(Y_pred_scaled)
Y_test = scaler.inverse_transform(Y_test_unscaled)

steps_to_predict = int(steps * test_size)
current_step = steps - steps_to_predict - 1
current_X = X[current_step]
Y_pred_open = np.zeros((steps_to_predict, out_size))
Y_pred_open[0] = scaler.inverse_transform(model.predict(current_X.reshape(1, -1)))
for i in range(1, steps_to_predict):
    u_raw = Y_pred_open[i - 1]
    u_scaled = scaler.transform(u_raw.reshape(1, -1)).flatten()
    prev_state = current_X
    new_state = np.tanh(W_in @ u_scaled + W_res @ prev_state + bias)
    current_X = (1 - alpha) * prev_state + alpha * new_state
    Y_pred_open[i] = scaler.inverse_transform(
        model.predict(current_X.reshape(1, -1))
    )

r_2 = r2_score(Y_test, Y_pred_open)
mse = root_mean_squared_error(Y_test, Y_pred_open)

r_2, mse

(-0.13066962422959585, 9.053320618469574)

In [487]:
in_size = 3  # W_in
out_size = (
    3  # In and out 3 since we have 3D data and next position output is also a 3D pos
)
res_size = 500  # W_res
sparsity = 0.05
spec_rad = 1.2  # Need to test this out tbh
alpha = (
    0.3  # 70 percent previous and 30 percent current - Like adam momentum nn methods
)
tau_steps = 1

rng = np.random.default_rng(42)
bias = rng.uniform(-0.1, 0.1, res_size)
W_in = rng.uniform(-0.1, 0.1, (res_size, in_size))
W_sparse = sparse.random(
    res_size,
    res_size,
    density=sparsity,
    format="csr",
    data_rvs=lambda n: rng.uniform(-1.0, 1.0, n),
    rng=rng,
)
eigenvalues, _ = eigs(W_sparse, k=1, which="LM")
largest_eigenvalue = np.abs(eigenvalues[0])
W_res = (
    W_sparse * (spec_rad / largest_eigenvalue) if largest_eigenvalue > 0 else W_sparse
)

X = np.zeros((steps + transient_steps_springs, res_size))
X[0] = np.tanh(0)
for i in range(1, steps + transient_steps_springs):
    u = lorentz_scaled[i - 1]
    prev_state = X[i - 1]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X[i] = (1 - alpha) * prev_state + alpha * new_state
X = X[transient_steps_springs:]

X_model_data = X[:-tau_steps]
Y_model_data = lorentz_scaled[transient_steps_springs + tau_steps :]
X_train, X_test = (
    X_model_data[: int(len(X_model_data) * (1 - test_size))],
    X_model_data[int(len(X_model_data) * (1 - test_size)) :],
)
Y_train, Y_test_unscaled = (
    Y_model_data[: int(len(Y_model_data) * (1 - test_size))],
    Y_model_data[int(len(Y_model_data) * (1 - test_size)) :],
)
model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)
Y_pred = scaler.inverse_transform(Y_pred_scaled)
Y_test = scaler.inverse_transform(Y_test_unscaled)

steps_to_predict = int(steps * test_size)
current_step = steps - steps_to_predict - 1
current_X = X[current_step]
Y_pred_open = np.zeros((steps_to_predict, out_size))
Y_pred_open[0] = scaler.inverse_transform(model.predict(current_X.reshape(1, -1)))
for i in range(1, steps_to_predict):
    u_raw = Y_pred_open[i - 1]
    u_scaled = scaler.transform(u_raw.reshape(1, -1)).flatten()
    prev_state = current_X
    new_state = np.tanh(W_in @ u_scaled + W_res @ prev_state + bias)
    current_X = (1 - alpha) * prev_state + alpha * new_state
    Y_pred_open[i] = scaler.inverse_transform(model.predict(current_X.reshape(1, -1)))

r_2 = r2_score(Y_test, Y_pred_open)
mse = root_mean_squared_error(Y_test, Y_pred_open)

r_2, mse

(-1.078697830487865, 12.231751667664076)

### Plots

In [494]:
fig = lorentz_plot(lorentz_dataset[-Y_pred.shape[0] :])
fig.show()
fig = lorentz_plot(Y_pred_open)
fig.show()

In [495]:
fig = r_2_plot(Y_test[:, 0], Y_pred_open[:, 0])
fig.show()
fig = r_2_plot(Y_test[:, 1], Y_pred_open[:, 1])
fig.show()
fig = r_2_plot(Y_test[:, 2], Y_pred_open[:, 2])
fig.show()

In [485]:
# vis.plot_pareto_front(study).show()